In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create artifacts directory for dashboard consumption
os.makedirs('../../backend/artifacts', exist_ok=True)

# Load the raw data
df = pd.read_csv('../../data/raw/online_shoppers_intention.csv')

print("=========================================")
print("  E-COMMERCE PURCHASE PREDICTION EDA")
print("=========================================\n")

# ---------------------------------------------------------
# Section 1: Basic Information & Missing Values
# ---------------------------------------------------------
print("--- Section 1: Basic Information ---")
print(df.info())

print("\nDuplicate Rows:", df.duplicated().sum())

# Missing Value Visualization
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap (Clean Dataset)")
plt.tight_layout()
plt.savefig('../../backend/artifacts/missing_values_heatmap.png')
plt.show()

# ---------------------------------------------------------
# Section 2: Target Distribution
# ---------------------------------------------------------
print("\n--- Section 2: Target Distribution ---")
plt.figure(figsize=(6, 4))
sns.countplot(x='Revenue', data=df, palette='Set2')
plt.title('Class Imbalance: Revenue (Target)')
plt.tight_layout()
plt.savefig('../../backend/artifacts/class_distribution.png')
plt.show()

print("Class Distribution (Percentages):")
print(df['Revenue'].value_counts(normalize=True) * 100)

# ---------------------------------------------------------
# Section 3: Feature Correlation
# ---------------------------------------------------------
print("\n--- Section 3: Feature Correlation ---")
# Convert target to int explicitly for correlation math
df['Revenue_Int'] = df['Revenue'].astype(int)

# Select numeric columns (now safely including the integer target)
numeric_cols = df.select_dtypes(include=['float64', 'int64', 'int32']).columns

corr_target = (
    df[numeric_cols]
    .corrwith(df["Revenue_Int"])
    .drop("Revenue_Int") # Drop self-correlation
    .sort_values(ascending=False)
)

print("\nTop Positive Correlations:")
print(corr_target.head(5))

print("\nTop Negative Correlations:")
print(corr_target.tail(5))

# Correlation Bar Plot (Target)
plt.figure(figsize=(8, 6))
corr_target.sort_values().plot(kind='barh', color='steelblue')
plt.title("Feature Correlation with Revenue")
plt.xlabel("Correlation Coefficient")
plt.tight_layout()
plt.savefig('../../backend/artifacts/correlation_plot.png')
plt.show()

# Full Numerical Feature Correlation Matrix
plt.figure(figsize=(12, 8))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title("Numerical Feature Correlation Matrix")
plt.tight_layout()
plt.savefig('../../backend/artifacts/correlation_heatmap.png')
plt.show()

# ---------------------------------------------------------
# Section 4: Outlier Analysis (Targeted)
# ---------------------------------------------------------
print("\n--- Section 4: Outlier Analysis ---")
outlier_cols = [
    'Administrative_Duration',
    'Informational_Duration',
    'ProductRelated_Duration',
    'PageValues'
]

plt.figure(figsize=(12, 8))
for i, col in enumerate(outlier_cols):
    plt.subplot(2, 2, i+1)
    sns.boxplot(y=df[col], color='coral')
    plt.title(f"Outliers in {col}")

plt.tight_layout()
plt.savefig('../../backend/artifacts/outlier_boxplots.png')
plt.show()

# ---------------------------------------------------------
# Section 5: Pairplot for Key Features (Sampled for Performance)
# ---------------------------------------------------------
print("\n--- Section 5: Pairplot for Key Features ---")
important_cols = ['PageValues', 'BounceRates', 'ExitRates', 'Revenue']

# Sample the dataframe to speed up pairplot rendering while maintaining distribution
sample_df = df.sample(min(3000, len(df)), random_state=42)

sns.pairplot(sample_df[important_cols], hue='Revenue', palette='husl', diag_kind='kde')
plt.suptitle('Pairplot of Top Predictors by Class (Sampled)', y=1.02)
plt.savefig('../../backend/artifacts/key_features_pairplot.png')
plt.show()

# ---------------------------------------------------------
# Section 6: Purchase Rates by Category
# ---------------------------------------------------------
print("\n--- Section 6: Purchase Rates by Category ---")

plt.figure(figsize=(16, 5))

# Purchase Rate by Visitor Type
plt.subplot(1, 3, 1)
purchase_rate_visitor = df.groupby("VisitorType")["Revenue_Int"].mean().sort_values(ascending=False)
purchase_rate_visitor.plot(kind="bar", color='skyblue')
plt.title("Purchase Rate by Visitor Type")
plt.ylabel("Conversion Rate")
plt.xticks(rotation=45)

# Purchase Rate by Month
plt.subplot(1, 3, 2)
purchase_rate_month = df.groupby("Month")["Revenue_Int"].mean().sort_values(ascending=False)
purchase_rate_month.plot(kind="bar", color='lightgreen')
plt.title("Purchase Rate by Month")
plt.ylabel("Conversion Rate")
plt.xticks(rotation=45)

# Purchase Rate by Weekend
plt.subplot(1, 3, 3)
weekend_rate = df.groupby("Weekend")["Revenue_Int"].mean()
weekend_rate.plot(kind="bar", color=["steelblue", "orange"])
plt.title("Purchase Rate by Weekend")
plt.ylabel("Conversion Rate")
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('../../backend/artifacts/purchase_rate_analysis.png')
plt.show()

# Clean up temporary column
df.drop('Revenue_Int', axis=1, inplace=True)

# ---------------------------------------------------------
# Section 7: Skewness Report
# ---------------------------------------------------------
print("\n--- Section 7: Feature Skewness Report ---")
skewness = df[numeric_cols.drop('Revenue_Int')].skew().sort_values(ascending=False)
print("\nMost Skewed Features:")
print(skewness.head(10))
print("(Highly skewed features will require log transformation prior to modeling)")

# ---------------------------------------------------------
# Section 8: Final EDA Summary
# ---------------------------------------------------------
print("\n=========================================")
print("EDA SUMMARY")
print("=========================================")
print("""
1. Dataset contains ~12k sessions.
2. No missing values detected.
3. Target is highly imbalanced (~84/16).
4. PageValues shows strongest positive correlation.
5. BounceRates and ExitRates show negative correlation.
6. Duration-based features contain heavy outliers.
7. Several features are highly skewed.
8. VisitorType and Month affect conversion rate.
9. Feature engineering required before modeling.
""")